# NACA 0012 Baseline Validation (Stage 4)

Compare orchestrator-driven OpenFOAM SA simulations of NACA 0012 at
$\alpha=0^\circ$ and $\alpha=10^\circ$ against the NASA Turbulence
Modeling Resource SA-model reference values:

| AoA | $C_L$ (NASA TMR) | $C_D$ (NASA TMR) |
|-----|------------------|------------------|
| 0°  | 0.0000           | 0.00819          |
| 10° | 1.0909           | 0.01231          |

**Tolerance per Stage-4 brief:** $\pm 2\%$ on $C_L$, $\pm 10\%$ on $C_D$
at $\alpha=10^\circ$. At $\alpha=0^\circ$ the relative-error metric is
undefined (lift goes to zero), so we apply the absolute bound
$|C_L| < 0.005$ instead.

**Data flow:** Stage 4's `scripts/smoke_naca0012.py` pulls
`coefficient.dat` from each run's persistent deploy directory on the
aero LXC into `results/01-naca0012-baseline/aoa-<deg>/`. This notebook
averages over the last 200 iterations and tabulates against the NASA
TMR reference.

In [ ]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd

RESULTS_DIR = Path('../results/01-naca0012-baseline')
AVERAGING_WINDOW = 200  # last N iterations

NASA_TMR = {
    0:  {'Cl': 0.0000, 'Cd': 0.00819},
    10: {'Cl': 1.0909, 'Cd': 0.01231},
}

In [ ]:
def load_coefficients(aoa_dir: Path) -> pd.DataFrame:
    '''Load coefficient.dat from a forceCoeffs1 output.

    OpenFOAM v2412 writes the coefficient.dat header with column names
    on a `#` line. Newer versions emit Cd, Cs, Cl, CmRoll, CmPitch, CmYaw.
    '''
    cand = aoa_dir / 'postProcessing' / 'forceCoeffs1' / '0' / 'coefficient.dat'
    if not cand.exists():
        # Some OF builds use plural 'coefficients.dat'.
        cand = aoa_dir / 'postProcessing' / 'forceCoeffs1' / '0' / 'coefficients.dat'
    if not cand.exists():
        raise FileNotFoundError(f'coefficient(s).dat not found under {aoa_dir}')
    return pd.read_csv(
        cand,
        sep=r'\s+',
        comment='#',
        names=['t', 'Cd', 'Cs', 'Cl', 'CmRoll', 'CmPitch', 'CmYaw'],
        engine='python',
    )

def average_final_n(df: pd.DataFrame, n: int = AVERAGING_WINDOW) -> tuple[float, float]:
    tail = df.tail(n)
    return float(tail['Cl'].mean()), float(tail['Cd'].mean())

In [ ]:
rows = []
for aoa in (0, 10):
    aoa_dir = RESULTS_DIR / f'aoa-{aoa}'
    df = load_coefficients(aoa_dir)
    cl, cd = average_final_n(df)
    ref = NASA_TMR[aoa]
    cl_err_pct = (cl - ref['Cl']) / ref['Cl'] * 100 if ref['Cl'] else float('nan')
    cd_err_pct = (cd - ref['Cd']) / ref['Cd'] * 100
    # Verdict.
    if aoa == 0:
        cl_pass = abs(cl) < 0.005
    else:
        cl_pass = abs(cl_err_pct) < 2.0
    cd_pass = abs(cd_err_pct) < 10.0
    rows.append({
        'AoA': aoa,
        'Cl': cl,
        'Cl (NASA)': ref['Cl'],
        'Cl err %': cl_err_pct,
        'Cd': cd,
        'Cd (NASA)': ref['Cd'],
        'Cd err %': cd_err_pct,
        'PASS': cl_pass and cd_pass,
    })
results = pd.DataFrame(rows)
results

In [ ]:
# Plot residual history for each AoA to confirm convergence.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, aoa in zip(axes, (0, 10)):
    df = load_coefficients(RESULTS_DIR / f'aoa-{aoa}')
    ax.plot(df['t'], df['Cl'], label='Cl')
    ax.plot(df['t'], df['Cd'] * 10, label='Cd x 10')
    ax.axhline(NASA_TMR[aoa]['Cl'], ls='--', color='C0', alpha=0.5, label='Cl (NASA)')
    ax.set_xlabel('iteration')
    ax.set_title(f'AoA = {aoa} deg')
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].set_ylabel('coefficient')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'force_history.png', dpi=120)

In [ ]:
# Persist the comparison table.
results.to_csv(RESULTS_DIR / 'results.csv', index=False)
print('PASS/FAIL summary:')
print(results.to_string(index=False))
overall_pass = bool(results['PASS'].all())
print()
print(f"OVERALL: {'PASS' if overall_pass else 'FAIL'}")